# J1 — Inspection complète du dataset HF AgentPublic/service-public.

Objectif :
  1. Confirmer la volumétrie totale (chunks + docs uniques).
  2. Mesurer la distribution `audience` et `theme` natifs.
  3. Pré-bâtir le mapping theme natif → 5 thèmes vie courante.
  4. Mesurer la longueur des champs textuels (chunk_text, text, context).
  5. Distribution chunks/doc (préparer Option D : reconstitution + 3 chunkings).
  6. Détecter d'éventuelles inconsistances de metadata par doc_id.


In [1]:
import random
from collections import Counter
from pathlib import Path
from statistics import median, mean

from datasets import load_dataset


DATASET_NAME = "AgentPublic/service-public"  # NB: nouveau nom (sans _fr)
CACHE_DIR = "data/raw"
SEED = 42
N_SAMPLES = 3


# Mapping initial (à raffiner après inspection des thèmes natifs)
# Heuristique : on cherche un mot-clé discriminant dans le theme natif.
THEME_KEYWORDS = {
    "famille": ["Famille"],
    "travail": ["Travail"],
    "logement": ["Logement"],
    "papiers_citoyennete": ["Papiers", "Citoyenneté", "Étrangers"],
    "argent_aides_impots": ["Argent", "Impôts", "Social"],
}

/home/stan/1.projects/rag-souverain-eval/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def map_native_theme(native_theme: str) -> str | None:
    """Mappe un theme natif vers l'un des 5 thèmes vie courante, ou None."""
    if not native_theme:
        return None
    # On prend juste le premier élément si liste de thèmes concaténée
    first_theme = native_theme.split(",")[0].strip()
    for target, keywords in THEME_KEYWORDS.items():
        if any(kw in first_theme for kw in keywords):
            return target
    return None

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print(f"=== Loading {DATASET_NAME} ===")
ds = load_dataset(DATASET_NAME, cache_dir=CACHE_DIR)

split_name = next(iter(ds.keys()))
data = ds[split_name]
print(f"Split: {split_name}, total chunks: {len(data)}")

=== Loading AgentPublic/service-public ===
Split: train, total chunks: 34944


## 1. Audience 

In [3]:
print("\n--- Audiences ---")
for aud, count in Counter(data["audience"]).most_common():
    pct = count * 100 // len(data)
    print(f"  {count:6d} ({pct:3d}%)  {aud}")


--- Audiences ---
   25599 ( 73%)  Particuliers
    8113 ( 23%)  Professionnels
    1232 (  3%)  Particuliers, Professionnels


## 2. Thèmes natifs (top 30)

In [4]:
print("\n--- Thèmes natifs (top 30 sur ~937) ---")
# On normalise : un seul thème par chunk (le premier, vu la duplication observée)
normalized_themes = [
    (t.split(",")[0].strip() if t else "")
    for t in data["theme"]
]
theme_counter = Counter(normalized_themes)
for theme, count in theme_counter.most_common(30):
    print(f"  {count:6d}  {theme}")


--- Thèmes natifs (top 30 sur ~937) ---
    5055  Travail - Formation
    3303  Famille - Scolarité
    3181  Argent - Impôts - Consommation
    3014  Logement
    2411  Justice
    1933  Social - Santé
    1765  Papiers - Citoyenneté - Élections
    1708  Étapes de vie
    1696  Transports - Mobilité
    1691  Étranger - Europe
    1558  Secteurs d'activité
    1454  Ressources humaines
    1356  Environnement
    1037  Fiscalité
     687  Associations
     587  Fonctionnement de l'entreprise
     571  Loisirs - Sports - Culture
     553  Pratiques commerciales
     392  Difficultés financières
     343  
     241  Baux
     238  Comptabilité - Facturation
     170  Financement


In [5]:
themes_split_list = [(([s.strip() for s in t.split(",")]) if t else "") for t in data.filter(lambda x: "Particuliers" in  x["audience"])["theme"]]
themes_flatten_list = [
    s
    for l in themes_split_list
    for s in l
]
unique_themes = set(themes_flatten_list)
print(len(unique_themes))
unique_themes

23


{'Argent - Impôts - Consommation',
 'Associations',
 'Baux',
 'Comptabilité - Facturation',
 'Difficultés financières',
 'Environnement',
 'Famille - Scolarité',
 'Financement',
 'Fiscalité',
 "Fonctionnement de l'entreprise",
 'Justice',
 'Logement',
 'Loisirs - Sports - Culture',
 'Papiers - Citoyenneté - Élections',
 'Pratiques commerciales',
 'Ressources humaines',
 "Secteurs d'activité",
 'Social - Santé',
 'Transports - Mobilité',
 'Travail - Formation',
 'fondations et fonds de dotation',
 'Étapes de vie',
 'Étranger - Europe'}

In [17]:
# Combien de chunks avec "Étapes de vie" comme seul thème ?
solo_etapes = sum(
    1 for t in data["theme"]
    if t and [x.strip() for x in t.split(",")] == ["Étapes de vie"]
)
print(f"Étapes de vie en solo : {solo_etapes*100/len(data):.1f} %")

Étapes de vie en solo : 0.2 %


## 3. Mapping vers les 5 thèmes vie courante 

In [6]:
print("\n--- Mapping natif → 5 thèmes vie courante ---")
mapped = Counter()
unmapped_samples = []
for native in normalized_themes:
    target = map_native_theme(native)
    if target:
        mapped[target] += 1
    else:
        mapped["__unmapped__"] += 1
        if len(unmapped_samples) < 20 and native:
            unmapped_samples.append(native)

for target, count in mapped.most_common():
    pct = count * 100 // len(data)
    print(f"  {count:6d} ({pct:3d}%)  {target}")

if unmapped_samples:
    print("\n  Échantillon de thèmes natifs NON mappés (à examiner) :")
    for t in sorted(set(unmapped_samples)):
        print(f"    - {t}")


--- Mapping natif → 5 thèmes vie courante ---
   16693 ( 47%)  __unmapped__
    5114 ( 14%)  argent_aides_impots
    5055 ( 14%)  travail
    3303 (  9%)  famille
    3014 (  8%)  logement
    1765 (  5%)  papiers_citoyennete

  Échantillon de thèmes natifs NON mappés (à examiner) :
    - Pratiques commerciales


## 4. Distribution chunks/doc 

In [7]:
print("\n--- Chunks par doc_id ---")
docs = Counter(data["doc_id"])
chunk_counts = sorted(docs.values())
print(f"  Documents uniques : {len(docs)}")
print(f"  Chunks/doc  min : {chunk_counts[0]}")
print(f"  Chunks/doc  p25 : {chunk_counts[len(chunk_counts)//4]}")
print(f"  Chunks/doc  median : {chunk_counts[len(chunk_counts)//2]}")
print(f"  Chunks/doc  p75 : {chunk_counts[3*len(chunk_counts)//4]}")
print(f"  Chunks/doc  p95 : {chunk_counts[int(0.95*len(chunk_counts))]}")
print(f"  Chunks/doc  max : {chunk_counts[-1]}")
print(f"  Chunks/doc  mean : {sum(chunk_counts)/len(chunk_counts):.1f}")


--- Chunks par doc_id ---
  Documents uniques : 3557
  Chunks/doc  min : 1
  Chunks/doc  p25 : 4
  Chunks/doc  median : 8
  Chunks/doc  p75 : 13
  Chunks/doc  p95 : 25
  Chunks/doc  max : 103
  Chunks/doc  mean : 9.8


## 5. Longueurs textuelles (échantillon) 

In [8]:
print("\n--- Longueurs (chars) sur échantillon de 2000 chunks ---")
sample_size = min(2000, len(data))
sample = data.select(range(sample_size))

for col in ["text", "chunk_text", "introduction"]:
    if col not in data.column_names:
        continue
    lens = [len(t) if t else 0 for t in sample[col]]
    print(
        f"  {col:14s} | "
        f"min={min(lens):5d} | "
        f"p50={sorted(lens)[len(lens)//2]:5d} | "
        f"mean={int(mean(lens)):5d} | "
        f"max={max(lens):6d}"
    )

# Profondeur du `context`
if "context" in data.column_names:
    ctx_depths = [len(c) if c else 0 for c in sample["context"]]
    print(
        f"  context depth   | "
        f"min={min(ctx_depths)} | "
        f"median={int(median(ctx_depths))} | "
        f"max={max(ctx_depths)}"
    )
    empty_ctx = sum(1 for d in ctx_depths if d == 0)
    print(f"  context vides   | {empty_ctx}/{sample_size} ({empty_ctx*100//sample_size}%)")


--- Longueurs (chars) sur échantillon de 2000 chunks ---
  text           | min=    2 | p50=  680 | mean=  969 | max=  4107
  chunk_text     | min=  239 | p50= 1030 | mean= 1341 | max=  4633
  introduction   | min=   48 | p50=  362 | mean=  350 | max=   552
  context depth   | min=0 | median=2 | max=3
  context vides   | 213/2000 (10%)


## 6. Sanity check metadata (constantes par doc_id ?) 

In [9]:
print("\n--- Sanity check : metadata constantes par doc_id ? ---")
import pandas as pd
sample_df = sample.to_pandas()
for col in ["title", "url", "audience"]:
    if col in sample_df.columns:
        nunique_per_doc = sample_df.groupby("doc_id")[col].nunique()
        max_variation = nunique_per_doc.max()
        n_inconsistent = (nunique_per_doc > 1).sum()
        print(f"  {col:14s} | max nunique/doc = {max_variation} | docs inconsistants = {n_inconsistent}")


--- Sanity check : metadata constantes par doc_id ? ---
  title          | max nunique/doc = 1 | docs inconsistants = 0
  url            | max nunique/doc = 1 | docs inconsistants = 0
  audience       | max nunique/doc = 1 | docs inconsistants = 0


## 7. Quelques exemples diversifiés 

In [10]:
print(f"\n--- {N_SAMPLES} exemples (seed={SEED}) ---")
random.seed(SEED)
sample_idx = random.sample(range(len(data)), N_SAMPLES)
for i, idx in enumerate(sample_idx, 1):
    record = data[idx]
    print(f"\n[Exemple {i} — idx={idx}]")
    for k in ["chunk_id", "doc_id", "chunk_index", "audience", "theme", "title", "url", "context"]:
        v = record.get(k)
        if isinstance(v, str) and len(v) > 150:
            v = v[:150] + "..."
        print(f"  {k}: {v}")
    # Tronquer les longs
    print(f"  text (200 chars): {(record.get('text') or '')[:200]}...")

print("\n=== Inspection terminée ===")



--- 3 exemples (seed=42) ---

[Exemple 1 — idx=7296]
  chunk_id: F1690_13
  doc_id: F1690
  chunk_index: 13
  audience: Particuliers
  theme: Travail - Formation
  title: Qu'est-ce que la disponibilité d'office pour raison de santé du fonctionnaire titulaire ?
  url: https://www.service-public.gouv.fr/particuliers/vosdroits/F1690
  context: ['FPT', "Quelle est la durée de la disponibilité d'office pour raison de santé ?"]
  text (200 chars): La durée de la disponibilité d'office pour raison de santé est d'un an maximum. Elle peut être renouvelée 2 fois pour la même durée. Si, à la fin de la 3 e année de disponibilité, vous êtes inapte à r...

[Exemple 2 — idx=1639]
  chunk_id: F115_6
  doc_id: F115
  chunk_index: 6
  audience: Particuliers
  theme: Justice, Justice
  title: Saisie sur salaire (ou "saisie des rémunérations")
  url: https://www.service-public.gouv.fr/particuliers/vosdroits/F115
  context: ['À partir du 1er juillet 2025', 'Pour quoi et comment rechercher un accord débite